# Introduction

This notebook will train a neural network model to generate metamath assertion statements.

# Imports

In [1]:
from pathlib import Path
import os
import random

import torch

from source.shared import Encoder
from source.shared import load_model
from source.shared import Parser01
from source.shared import SyntaxDeriverValidationReporter

from source.create_files import create_files
from source.create_model import create_model
from source.train_model import train_model
from source.train_model import plot_correct
from source.evaluate_model import ModelEvaluator

!python --version
!which python
# !where python
# !conda list

Python 3.12.8
/opt/anaconda3/envs/python312/bin/python


# Settings

In [2]:
import param
import panel as pn
pn.extension()
pn.config.sizing_mode="stretch_width"

block_size = 150  # what is the maximum context length for predictions?
learning_rate = 1e-4
n_embd = 1000
dropout = 0.2

class Settings(param.Parameterized):
    limit_count = param.Integer(1000 * 40, allow_None=True)

    n_embd = param.Integer(1000, label="n_embd")
    dropout = param.Number(0.2, label="dropout")
    n_head = param.Integer(10, label='n_head') # default 10
    block_size = param.Integer(150, label='block_size')  # the maximum context length for predictions
    n_layer = param.Integer(10, label='n_layer')

    learning_rate = param.Number(1e-4,label='learning_rate')

    mmx_file_path = param.Path(default=os.fspath(Path('set.new2023.mmx').resolve()), label='mmx_file_path')
    corpus01_file_path = param.Path(default=os.fspath(Path('corpus01.txt').resolve()), check_exists=False, label='corpus01_file_path')
    corpus_folder_path = param.Path(default=os.fspath(Path('corpus').resolve()), label='corpus_folder_path')
    model_folder_path = param.Path(default=os.fspath(Path("model").resolve()), label='model_folder_path')

    def view(self):
        return pn.WidgetBox(
            pn.Column(pn.pane.Markdown("# Settings")),
            pn.Column(
                "## Parser03",
                self.param.limit_count,
            ),
            pn.Column(
                "## model",
                self.param.n_embd,
                self.param.n_head,
                self.param.block_size,
                self.param.dropout,
                self.param.n_layer,
            ),
            pn.Column(
                "## optimizer",
                self.param.learning_rate,
            ),
            pn.Column(
                "## paths",
                self.param.mmx_file_path,
                self.param.corpus01_file_path,
                self.param.corpus_folder_path,
                self.param.model_folder_path,
            ),
            pn.Column(pn.pane.Markdown("# End")),
            max_width=600,
        )

settings = Settings()
settings.view()

WidgetBox(max_width=600, sizing_mode='stretch_width')
    [0] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
    [1] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] IntInput(name='Limit count', sizing_mode='stretch_width', value=40000)
    [2] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] IntInput(name='n_embd', sizing_mode='stretch_width', value=1000)
        [2] IntInput(name='n_head', sizing_mode='stretch_width', value=10)
        [3] IntInput(name='block_size', sizing_mode='stretch_width', value=150)
        [4] FloatInput(name='dropout', sizing_mode='stretch_width', value=0.2)
        [5] IntInput(name='n_layer', sizing_mode='stretch_width', value=10)
    [3] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] FloatInput(name='learning_rate', sizing_mode='stretch_width', value=0.0001)
    [4] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] LiteralInput(name='mmx_file_path', sizing_mode='stretch_width', value='/Users/hale/PycharmProjec...)
        [2] LiteralInput(name='corpus01_file_path', sizing_mode='stretch_width', value='/Users/hale/PycharmProjec...)
        [3] LiteralInput(name='corpus_folder_path', sizing_mode='stretch_width', value='/Users/hale/PycharmProjec...)
        [4] LiteralInput(name='model_folder_path', sizing_mode='stretch_width', value='/Users/hale/PycharmProjec...)
    [5] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')

# Create corpus01.txt

In [3]:
print(f"=== create corpus01.txt ===")
def set_up_corpus01():
    corpus01_file_path = Path(settings.corpus01_file_path)
    if os.path.exists(corpus01_file_path):
        print(f"corpus01.txt already exists: {corpus01_file_path}")
    else:
        mmx_file_path = Path(settings.mmx_file_path)
        with open(mmx_file_path, "r") as mmx_file_object:
            mmx_file = mmx_file_object.read()
        Parser01(source=mmx_file).parse().save(corpus01_file_path)

set_up_corpus01()

=== create corpus01.txt ===
corpus01.txt already exists: /Users/hale/PycharmProjects/MathAssertGPT/corpus01.txt


In [4]:
# Get lines of corpus01 file to be examined.
with open(Path(settings.corpus01_file_path), 'r') as file:
    corpus01_lines = file.read().split('\n')
print(f'number_of_corpus01_lines={len(corpus01_lines): ,}')

number_of_corpus01_lines= 164,899


In [5]:
# Examine some lines of corpus01 file.
# Each line is essentially statements in the metamath database.
# Note that the typecode is put before its label for programming convenience.
random_corpus01_lines = random.sample(corpus01_lines[:1000], 3)
for random_corpus01_line in random_corpus01_lines:
    print(random_corpus01_line)
    print('-' * 80)

$e syldd.2 |- ( ph -> ( ps -> ( th -> ta ) ) ) $.
--------------------------------------------------------------------------------
$e pm5.74d.1 |- ( ph -> ( ps -> ( ch <-> th ) ) ) $.
--------------------------------------------------------------------------------
$e con1bii.1 |- ( -. ph <-> ps ) $.
--------------------------------------------------------------------------------


# Create corpus.txt encoder.json assert.db

In [6]:
print(f"=== create assert.db corpus.txt encoder.json ===")
def set_up_corpus() -> Path:
    corpus01_file_path = Path(settings.corpus01_file_path)
    limit_count = settings.limit_count
    mmx_file_path = Path(settings.mmx_file_path)
    corpus_file_path = Path(settings.corpus_folder_path).joinpath('corpus.txt').resolve()
    if corpus_file_path.exists():
        print(f"corpus.txt already exists: {corpus_file_path}")
    else:
        create_files(corpus_folder_path=settings.corpus_folder_path, limit_count=limit_count, mmx_file_path=mmx_file_path, corpus01_file_path=corpus01_file_path)
    return corpus_file_path

corpus_file_path = set_up_corpus()

=== create assert.db corpus.txt encoder.json ===
mmx_file_path=/Users/hale/PycharmProjects/MathAssertGPT/set.new2023.mmx
10000: equsalhw
20000: $p
30000: dmsnn0
40000: suppssof1
['CUQAEFVBDQURSUSUT', '$.']
#assert_corpus=43457
AssertDB did commit
assert_corpus_size=43457
vocab_size=160
create corpus.txt: size=43457; corpus_file_path=/Users/hale/PycharmProjects/MathAssertGPT/corpus/corpus.txt
create encoder.txt: corpus_folder_path=/Users/hale/PycharmProjects/MathAssertGPT/corpus
AssertDB did commit
AssertB did close
Done create_files


In [7]:
# Get lines of corpus file to be examined.
with open(Path(corpus_file_path), 'r') as file:
    corpus_lines = file.read().split('\n')
print(f'number_of_corpus_lines={len(corpus_lines): ,}')

number_of_corpus_lines= 43,458


In [8]:
# Examine some lines of corpus file.
# Each line is essentially some assertion in some proof of the metamath database.
# The model will be training on these lines of the corpus file.
random_corpus_lines = random.sample(corpus_lines[:1000], 3)
for random_corpus_line in random_corpus_lines:
    print(random_corpus_line)
    print('-' * 80)

|- ( X F ( F ` X ) \/ ( F ` X ) = (/) ) <|over|>
--------------------------------------------------------------------------------
|- ( A C_ ( B u. C ) <-> A C_ ( C u. B ) ) <|over|>
--------------------------------------------------------------------------------
|- ( ps -> ( ( ch /\ ph ) -> ta ) ) <|over|>
--------------------------------------------------------------------------------


# Load model

In [9]:
# load model
def set_up_model() -> Path:
    model_folder_path = Path(settings.model_folder_path)
    model_name = 'model.pt'
    model_file_path = model_folder_path.joinpath(model_name).resolve()
    if model_file_path.exists():
        print(f"model_file already exists: {model_file_path}")
    else:
        create_model(settings=settings)
    return model_file_path

model_file_path = set_up_model()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.backends.mps.is_available():
    device = "mps"
print(f'device={device}')
encoder = Encoder.load_from_json(corpus_folder_path=settings.corpus_folder_path)
print(f'loading model and optimizer from checkpoint={model_file_path}')
model, optimizer = load_model(model_checkpoint_path=model_file_path, device=device, encoder=encoder)

settings.block_size = 150
Start create model and optimizer
create_model: model_file_path=/Users/hale/PycharmProjects/MathAssertGPT/model/model.pt
create_encoder
vocab_size=160
create model: block_size=150 device=mps
120.57216 M parameters
create a PyTorch optimizer
model created at path=/Users/hale/PycharmProjects/MathAssertGPT/model/model.pt
Done
device=mps
loading model and optimizer from checkpoint=/Users/hale/PycharmProjects/MathAssertGPT/model/model.pt


# Train model

In [10]:
%%time
# train model
max_train_epochs = 10 * 10 * 1 * 1 * 1
train_model(model, optimizer, max_train_epochs=max_train_epochs, corpus_folder_path=settings.corpus_folder_path, model_folder_path=settings.model_folder_path)

vocab_size=160
corpus_file_path=/Users/hale/PycharmProjects/MathAssertGPT/corpus/corpus.txt
corpus_statement_count=43457
#encoded_train_statements=43457
=== train and evalate model ===
epoch=0; step=0; n_head=10; n_layer=10
train_batch_size=10
max_train_epochs=100 eval_interval=10
block_size=150 n_head=10 n_layer=10 learning_rate=0.0001 device=mps
#train_dataset=43457
       0. train loss 6.3373
      10. train loss 0.6848
      20. train loss 0.6634
      30. train loss 0.5244
      40. train loss 0.6536
      50. train loss 0.4087
      60. train loss 0.8401
      70. train loss 0.6807
      80. train loss 0.7573
      90. train loss 0.6799
      99. train loss 0.5032
elapsed_time=0.68 minutes
epoch=100; step=1000
saving model: epoch=100 model_checkpoint_path=/Users/hale/PycharmProjects/MathAssertGPT/model/model.pt
CPU times: user 56.7 s, sys: 3min 37s, total: 4min 33s
Wall time: 53.3 s


# Evaluate model

In [ ]:
%%time
def print_context(context):
    parts = context.split('\n')
    if len(parts) >= 2:
        print(f'prompt: {parts[0]}')
        print(f'predicted_statement: {parts[1]}')

def evaluate_model(model, max_examples: int, max_print_error: int, max_print_ok: int):
    model_evaluator = ModelEvaluator(corpus_folder_path=settings.corpus_folder_path, model=model)
    model_evaluator.evaluate_model(max_examples=max_examples)
    syntax_deriver_db= model_evaluator.syntax_deriver.syntax_deriver_db
    validation_reporter = SyntaxDeriverValidationReporter(syntax_deriver_db=syntax_deriver_db, block_size=150)
    validation_reporter.print_validation_report(max_print_error=max_print_error, max_print_ok=max_print_ok, print_context=print_context)

    # print(f'\n=== Show all tables ===')
    # syntax_deriver = model_evaluator.syntax_deriver
    # syntax_deriver.syntax_deriver_db.show_math_statements_table()
    # syntax_deriver.syntax_deriver_db.show_rule_errors_table()
    # print(f'\n=== Show main_view ===')
    # syntax_deriver.syntax_deriver_db.create_main_view()
    # syntax_deriver.syntax_deriver_db.print_main_view()

max_examples = 10 * 1 * 1 * 1
max_print_error = 3
max_print_ok = 0
if max_examples > 0:
    evaluate_model(model=model, max_examples=max_examples, max_print_error=max_print_error, max_print_ok=max_print_ok)

# Plot

In [ ]:
plot_correct(model_folder_path=settings.model_folder_path, bucket_count=50, xlabel='train step', ylabel='percent correct', title='assert logic')

# Inference

In [ ]:
# This is not needed for doing inference.
# It is used to see if the inference is not in the train examples.
with open(corpus_file_path, 'r') as file:
    train_examples = file.read().split('\n')
set_of_train_examples = set(train_examples)
print(f'#train_examples={len(train_examples)} #set_of_train_examples={len(set_of_train_examples)}')

In [ ]:
# Print a train example.
print(f'train_example: {train_examples[100]}')

In [ ]:
# This is needed for doing inference.
# This cell should be run only once (but does not matter if run multiple times).
from source.shared import generate_predicted_dictum

prompt = '|- '
terminal_token = '<|over|>'

# Put model in inference mode (not training mode).
_ = model.eval()

In [ ]:
%%time
# Make an inference using the model.
# The predicted dictum may not be a valid metamath statement.
# The time to make the inference may not be fast (maybe due to coding inefficiency).
# You can run this cell repeatedly to get different inferences.
predicted_dictum = generate_predicted_dictum(prompt=prompt, terminal_token=terminal_token, model=model)
print(f'predicted_dictum: {predicted_dictum}')
print(f'predicted_dictum is in train examples: {predicted_dictum in set_of_train_examples}')

# New statement percentage

In [ ]:
%%time
new_count = 0
max_statement_count = 10 * 1
for _ in range(max_statement_count):
    predicted_dictum = generate_predicted_dictum(prompt=prompt, terminal_token=terminal_token, model=model)
    if predicted_dictum not in set_of_train_examples:
        new_count += 1
print(f'new_statement_percentage={100 * new_count / max_statement_count: .0f}%')